# preparing tables and figures for the thesis

In [2]:
import pandas as pd
import numpy as np

In [4]:
atlas_csv = pd.read_csv("/home/gaia/Projects/legacy_data/my_master/space-MNI152_atlas-schaefer2018tian2020_res-1mm_den-400_div-7networks_dseg.csv")

coef_df = pd.read_csv('/home/gaia/Projects/legacy_data/legacy_pipe/data/interim/coef_df_manual_bins_wuniform.csv')

# take only interesting bins: (20, 25], (25, 30], (30, 35]
coef_df = coef_df[coef_df['age_bin'].isin(['(20, 25]', '(25, 30]', '(30, 35]'])]

unweighted_birth_year_coef_df = coef_df[
    (coef_df['variable'] == 'birth_year') & (coef_df['weighting'] == 'unweighted')
]
weighted_birth_year_coef_df = coef_df[
    (coef_df['variable'] == 'birth_year') & (coef_df['weighting'] == 'weighted')
]
# rename the column 'index' to 'region_label'
atlas_csv = atlas_csv.rename(columns={'index': 'region_label'})

# add the network and component information to the birth_year_coef_df
unweighted_birth_year_coef_df = unweighted_birth_year_coef_df.merge(atlas_csv[['region_label', 'network', 'component', 'hemisphere']], on='region_label', how='left')
weighted_birth_year_coef_df = weighted_birth_year_coef_df.merge(atlas_csv[['region_label', 'network', 'component', 'hemisphere']], on='region_label', how='left')

In [8]:
col_list = ['region_label', 'region_name', 'coef', 't', 'fdr_p' ]
dec = 3

for age_bin in coef_df['age_bin'].unique():
    sub_df = weighted_birth_year_coef_df[weighted_birth_year_coef_df['age_bin'] == age_bin]
    print(f"Age bin: {age_bin}, n_subjects = {sub_df['n_subjects'].iloc[0]}, ess = {sub_df['ess'].iloc[0]}")
    # keep columns that are in col_list
    sub_df = sub_df[col_list]

    # keep only dec numbers after the decimal point
    for col in sub_df.columns:
        if col in ['coef', 't', 'fdr_p']:
            sub_df[col] = sub_df[col].round(dec)
    # save table so I can export to csv
    sub_df.to_csv(f"/home/gaia/Projects/legacy_data/legacy_pipe/docs/weighted_birth_year_coef_df_{age_bin}.csv", index=False)


Age bin: (20, 25], n_subjects = 1068, ess = 279.8028769116858
Age bin: (25, 30], n_subjects = 1116, ess = 485.3667022802616
Age bin: (30, 35], n_subjects = 595, ess = 245.1840347411026
